<h1> Hands on: TDDFT with <span style="color: red;">QE</span>py</h1>

#### Obtain the Optical Absorption of Molecules with real-time TDDFT (ce-tddft)


## To run this tutorial we need to
 - import QEpy
 - Use ASE to generate molecular structures

In [ ]:
# --- optional pip installs (normally leave collapsed / do not run) ---
!pip install qepy f90wrap==0.2.16
!pip install dftpy
!pip install matplotlib

In [2]:
import qepy
from qepy.driver import Driver
from qepy.io import QEInput
from ase.build import molecule
import numpy as np
import matplotlib.pyplot as plt

## Generate the QE options to run a scf calculation

In [3]:
prefix = 'c2h4'

In [4]:
import os
print(os.getcwd())
outdir = os.path.abspath('tmp')

/Users/aniketmandal/qepy_paper_nb


In [5]:
scf_options = {}

scf_options['&control'] = {}
scf_options['&control']['calculation'] = 'scf'
scf_options['&control']['prefix'] = prefix
scf_options['&control']['restart_mode'] = 'from_scratch'
scf_options['&control']['outdir'] = outdir
scf_options['&control']['pseudo_dir'] = './data'

scf_options['&system'] = {}
scf_options['&system']['ecutwfc'] = 25

scf_options['&electrons'] = {}
scf_options['&electrons']['conv_thr'] = 1.0E-10
#scf_options['&electrons']['diagonalization'] = 'cg' #Use if SCF fails to converge

scf_options['atomic_species'] = []
scf_options['atomic_species'].append('H  1.008  H.pz-vbc.UPF')
scf_options['atomic_species'].append('C 12.011  C.pz-vbc.UPF')

scf_options['k_points {gamma}'] = []

## Workflow for SCF

 1. Use ASE to build a new molecule
 2. Set periodic cell
 3. Update  QEpy options dictionary
 4. Initialize QEpy `Driver` class
 5. Run SCF
 6. Save wavefunction and eigenvalues data

## Steps 1-3: Build molecule, set cell and generate options dictionary

In [6]:
atoms = molecule ("C2H4")
atoms.set_cell(cell=np.identity(3)*10)
atoms.translate([5,5,5])
pwin = QEInput()
scf_options = pwin.update_atoms(atoms, qe_options=scf_options)

In [7]:
from pprint import pprint
pprint(scf_options)

{'&control': {'calculation': 'scf',
              'outdir': '/Users/aniketmandal/qepy_paper_nb/tmp',
              'prefix': 'c2h4',
              'pseudo_dir': './data',
              'restart_mode': 'from_scratch'},
 '&electrons': {'conv_thr': 1e-10},
 '&system': {'ecutwfc': 25, 'ibrav': 0, 'nat': 6, 'ntyp': 2},
 'atomic_positions angstrom': ['C    5.00000000000000 5.00000000000000 '
                               '5.66748000000000',
                               'C    5.00000000000000 5.00000000000000 '
                               '4.33252000000000',
                               'H    5.00000000000000 5.92283200000000 '
                               '6.23769500000000',
                               'H    5.00000000000000 4.07716800000000 '
                               '6.23769500000000',
                               'H    5.00000000000000 5.92283200000000 '
                               '3.76230500000000',
                               'H    5.00000000000000 4.07716800

## Step 4: Initialize the Driver class with the scf QE options

In [8]:
driver = Driver(qe_options=scf_options, task='scf', logfile=prefix+'.scf.out')

## Step 5: Run a scf calculation

In [9]:
driver.scf()

-26.85622518626664

## Step 6: Analyze: save the wavefunction for follow-up TDDFT calculation

In [10]:
driver.save()

In [11]:
driver.stop()

# Real-time TDDFT optical spectrum of a molecule

 1. Generate appropriate real-time TDDFT qe options
 2. Initialize driver
 3. Propagate in real time and collect the dipole
 4. Analyze data (plot optical spectrum)

## Step 1: Generate the QE options for the real-time TDDFT calculation

In [10]:
tddft_options = {}
tddft_options["&inputtddft"] = {}

In [11]:
tddft_options["&inputtddft"]["job"] = 'optical'
tddft_options["&inputtddft"]["prefix"] = prefix
tddft_options["&inputtddft"]["tmp_dir"] = 'tmp'      # must match the scf outdir
tddft_options["&inputtddft"]["dt"] = 2.0             # time step
tddft_options["&inputtddft"]["nstep"] = 2000         # # of propagation steps (raise for resolution)
tddft_options["&inputtddft"]["e_direction"] = 1      # kick polarization: 1/2/3 = x/y/z

In [1]:
ls -la tmp/c2h4.save/data-file-schema.xml

-rw-r--r--@ 1 aniketmandal  staff  38816 Aug  7 15:21 tmp/c2h4.save/data-file-schema.xml


In [3]:
!head -5 tmp/c2h4.save/data-file-schema.xml     # look for the <creator> / version tag
import qepy; print(qepy.__version__)

<?xml version="1.0" encoding="UTF-8"?>
<!-- All quantities are in Hartree atomic units unless otherwise specified -->
<qes:espresso xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:qes="http://www.quantum-espresso.org/ns/qes/qes-1.0" xsi:schemaLocation="http://www.quantum-espresso.org/ns/qes/qes-1.0 http://www.quantum-espresso.org/ns/qes/qes_230310.xsd" Units="Hartree atomic units">
  <general_info>
    <xml_format NAME="QEXSD" VERSION="23.03.10">QEXSD_23.03.10</xml_format>
7.2.1rc0


## Step 2: Initialize the Driver class with the tddft QE options

In [ ]:
driver_td = Driver(qe_options=tddft_options, task='optical',
                   iterative=True, logfile=prefix+'.tddft.out')

## Step 3: Propagate in real time and collect the dipole

In [ ]:
nstep = tddft_options["&inputtddft"]["nstep"]
mu_t = np.zeros((nstep, 3))
for i in range(nstep):
    driver_td.diagonalize()                  # advance one real-time step
    mu_t[i] = driver_td.get_dipole_tddft()   # dipole at this step

In [ ]:
driver_td.stop()

# Step 4: Now we can plot the absorption spectra 

In [ ]:
# Fourier-transform the dipole response into an absorption spectrum
dt = tddft_options["&inputtddft"]["dt"]
e_strength, damping = 0.01, 0.02
ATTO, EV_J, HBAR = 1e-18, 1.602176634e-19, 1.054571817e-34

mu = mu_t[:, 0] - mu_t[0, 0]                        # x-component response
t  = np.arange(mu.size) * dt * ATTO * EV_J / HBAR   # -> inverse-eV time
w  = np.exp(-t**2 * damping)

E = np.linspace(0, 20, 2000)                        # eV
S = np.array([np.sum(mu * w * np.sin(e * t)) * e for e in E])
S *= 2 * dt * ATTO * EV_J / HBAR / (e_strength * np.pi)

In [ ]:
plt.plot(E, S)
plt.xlabel(r'$\omega$ (eV)')
plt.ylabel(r'Intensity (arb. units)')
plt.xlim([0,20])